<a href="https://colab.research.google.com/github/adityaghai07/ag-yt-lectures/blob/main/PQ_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

In [ ]:
np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

# Generate 1000 vectors of 8 dimensions
# In reality this would be embeddings from your model
data = np.random.rand(1000, 8).astype(np.float32)

print(f"Dataset: {data.shape[0]} vectors, {data.shape[1]} dimensions")
print(f"Raw size: {data.nbytes} bytes")

Dataset: 1000 vectors, 8 dimensions
Raw size: 32000 bytes


In [ ]:
data[0]

array([0.375, 0.951, 0.732, 0.599, 0.156, 0.156, 0.058, 0.866],
      dtype=float32)

In [ ]:
# PQ parameters
m = 4        # number of sub-vectors
k = 4        # centroids per codebook (in production this is 256)
d = 8        # total dimensions
ds = d // m  # dimensions per sub-vector = 2

# Split all vectors into sub-vectors
sub_vectors = [data[:, i*ds:(i+1)*ds] for i in range(m)]

print(f"\nSplit into {m} sub-vectors of {ds} dims each")
print(f"Sub-vector 0 shape: {sub_vectors[0].shape}")
print(f"First vector's sub-vectors:")
for i, sv in enumerate(sub_vectors):
    print(f"  Position {i}: {sv[0]}")


Split into 4 sub-vectors of 2 dims each
Sub-vector 0 shape: (1000, 2)
First vector's sub-vectors:
  Position 0: [0.375 0.951]
  Position 1: [0.732 0.599]
  Position 2: [0.156 0.156]
  Position 3: [0.058 0.866]


In [ ]:
# Train codebooks
codebooks = []
for i in range(m):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(sub_vectors[i])
    codebooks.append(km.cluster_centers_)
    print(f"\nCodebook {i} (dims {i*ds} to {(i+1)*ds - 1}):")
    for j, centroid in enumerate(km.cluster_centers_):
        print(f"  C{j}: {centroid}")


Codebook 0 (dims 0 to 1):
  C0: [0.721 0.237]
  C1: [0.771 0.743]
  C2: [0.273 0.735]
  C3: [0.231 0.222]

Codebook 1 (dims 2 to 3):
  C0: [0.716 0.721]
  C1: [0.233 0.782]
  C2: [0.258 0.268]
  C3: [0.765 0.246]

Codebook 2 (dims 4 to 5):
  C0: [0.254 0.296]
  C1: [0.754 0.729]
  C2: [0.244 0.786]
  C3: [0.75 0.23]

Codebook 3 (dims 6 to 7):
  C0: [0.751 0.761]
  C1: [0.748 0.258]
  C2: [0.243 0.736]
  C3: [0.221 0.232]


In [ ]:
# Encode a specific vector
x = data[0]
print(f"\nOriginal vector: {x}")
print(f"Original size: {x.nbytes} bytes")

codes = []
for i in range(m):
    sub = x[i*ds:(i+1)*ds]
    distances = np.linalg.norm(codebooks[i] - sub, axis=1)
    code = np.argmin(distances)
    codes.append(code)
    print(f"  Sub-vector {i}: {sub} → nearest C{code}={codebooks[i][code]} (dist={distances[code]:.4f})")

codes = np.array(codes, dtype=np.uint8)
print(f"\nEncoded: {codes}")
print(f"Compressed size: {codes.nbytes} bytes")
print(f"Compression ratio: {x.nbytes / codes.nbytes}x")


Original vector: [0.375 0.951 0.732 0.599 0.156 0.156 0.058 0.866]
Original size: 32 bytes
  Sub-vector 0: [0.375 0.951] → nearest C2=[0.273 0.735] (dist=0.2385)
  Sub-vector 1: [0.732 0.599] → nearest C0=[0.716 0.721] (dist=0.1239)
  Sub-vector 2: [0.156 0.156] → nearest C0=[0.254 0.296] (dist=0.1711)
  Sub-vector 3: [0.058 0.866] → nearest C2=[0.243 0.736] (dist=0.2258)

Encoded: [2 0 0 2]
Compressed size: 4 bytes
Compression ratio: 8.0x


In [ ]:
# Now the query and lookup table trick
query = np.random.rand(8).astype(np.float32)
print(f"\nQuery vector: {query}")

# Build lookup tables
tables = np.zeros((m, k))
for i in range(m):
    q_sub = query[i*ds:(i+1)*ds]
    for j in range(k):
        diff = q_sub - codebooks[i][j]
        tables[i][j] = np.sum(diff ** 2)

print("\nLookup tables (squared L2 distances):")
for i in range(m):
    print(f"  Table {i}: {tables[i]}")

# Score using only the codes
approx_dist = sum(tables[i][codes[i]] for i in range(m))
exact_dist = np.sum((query - x) ** 2)

print(f"\nApproximate distance (PQ): {approx_dist:.4f}")
print(f"Exact distance:           {exact_dist:.4f}")
print(f"Error:                    {abs(approx_dist - exact_dist):.4f}")


Query vector: [0.72  0.687 0.096 0.923 0.568 0.364 0.757 0.257]

Lookup tables (squared L2 distances):
  Table 0: [0.202 0.006 0.202 0.456]
  Table 1: [0.425 0.039 0.455 0.906]
  Table 2: [0.103 0.168 0.284 0.051]
  Table 3: [0.254 0.    0.493 0.287]

Approximate distance (PQ): 1.2232
Exact distance:           1.7704
Error:                    0.5472
